# Task 1 — Sentiment Analysis

Compare TF-IDF models with a pretrained transformer on ONE shared test split. Report accuracy, per-class precision/recall/F1 and the confusion matrix, then save our TF-IDF v2 model to `models/tfidf-v2/` for Hugging Face.

*Roman Urdu: Saare models ek hi test set pe compare honge, phir hamara model `models/tfidf-v2/` mein save hoga.*

In [16]:
# All imports in one place
import sys
sys.path.append("..")        # project root -> src/
sys.path.append("../app")    # app/ -> text_cleaning.py

import importlib
import json
import re
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import nltk
import pandas as pd
import sklearn
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_sample_weight

from src import config, preprocessing
import text_cleaning

importlib.reload(preprocessing)
importlib.reload(text_cleaning)      # pick up edits to app/text_cleaning.py

pre_processed = preprocessing.load_pre_processed()
pre_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27863 entries, 0 to 27862
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            27863 non-null  object 
 1   categories      27863 non-null  object 
 2   reviews.rating  27863 non-null  float64
 3   reviews.text    27863 non-null  object 
 4   reviews.title   27863 non-null  object 
dtypes: float64(1), object(4)
memory usage: 1.1+ MB


In [17]:
# Settings used everywhere below
RANDOM_STATE = 42
TEST_SIZE = 0.3
LABELS = ["negative", "neutral", "positive"]
RATING_TO_LABEL = {1.0: "negative", 2.0: "negative", 3.0: "neutral", 4.0: "positive", 5.0: "positive"}

y = pre_processed["reviews.rating"].map(RATING_TO_LABEL)
X = pre_processed["reviews.text"].astype(str)                                     # text only (TF-IDF v1)
X_full = pre_processed["reviews.title"].fillna("").astype(str) + ". " + X         # title + text (v2, transformer)

# ONE split, reused by every model, so all models are tested on the same reviews
train_idx, test_idx = train_test_split(
    pre_processed.index, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_train, X_test = X.loc[train_idx], X.loc[test_idx]
X_full_train, X_full_test = X_full.loc[train_idx], X_full.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)
print(y_train.value_counts(normalize=True).round(3))

# One helper to score every model the same way; results feed the comparison table
results = {}

def evaluate(name, y_true, y_pred):
    f1 = f1_score(y_true, y_pred, labels=LABELS, average=None)
    results[name] = {
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "macro_f1": round(f1.mean(), 3),
        "f1_negative": round(f1[0], 3),
        "f1_neutral": round(f1[1], 3),
        "f1_positive": round(f1[2], 3),
    }
    print(f"=== {name}")
    print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))
    print(confusion_matrix(y_true, y_pred, labels=LABELS))

Train: (19504,) Test: (8359,)
reviews.rating
positive    0.930
neutral     0.046
negative    0.024
Name: proportion, dtype: float64


In [18]:
# Step 1: remove HTML (same function the app uses, imported from app/text_cleaning.py)
from text_cleaning import remove_html_tags

print(remove_html_tags("<b>Great</b> tablet<br>for kids"))

 Great  tablet for kids


In [19]:
# Step 2: lowercase, letters only, drop single letters and extra spaces
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\b[a-z]\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()

print(clean_text("Great!! 10/10, I'd buy it again"))

great buy it again


In [20]:
# Step 3: remove stopwords (the list is built ONCE, not on every call)
STOP_WORDS = set(stopwords.words("english"))

def remove_stopwords(text):
    return " ".join(w for w in text.split() if w not in STOP_WORDS)

print(remove_stopwords("this is not a good tablet"))   # note: "not" is removed here (fixed in v2)

good tablet


In [21]:
# Step 4: lemmatize, then run all 4 steps ONCE on train and test
lemmatizer = WordNetLemmatizer()

def lemmatize(text):
    return " ".join(lemmatizer.lemmatize(w) for w in text.split())

def clean_v1(text):
    return lemmatize(remove_stopwords(clean_text(remove_html_tags(text))))

X_train_clean = X_train.apply(clean_v1)
X_test_clean = X_test.apply(clean_v1)
X_train_clean.head()

4223     love kindle kid love playing disney movie also...
23139    awesome better expected cannot live without alexa
16071    daughter love tablet amazon plenty game play a...
19850    value performance ease use lightweight could want
684      work great prety fast operating system dosent ...
Name: reviews.text, dtype: object

In [22]:
# TF-IDF: turn text into word scores, then show the most typical words per class
def top_terms(label, tfidf_matrix, vectorizer, top_n=5):
    """Words with the highest average TF-IDF score among training reviews of one label."""
    rows = (y_train == label).to_numpy()
    mean_scores = tfidf_matrix[rows].mean(axis=0).A1
    return pd.Series(mean_scores, index=vectorizer.get_feature_names_out()).nlargest(top_n)

def fit_tfidf(vectorizer):
    """Fit on train only, transform test, print shapes and top words per class."""
    X_tr = vectorizer.fit_transform(X_train_clean)
    X_te = vectorizer.transform(X_test_clean)
    print("Train:", X_tr.shape, "Test:", X_te.shape)
    for label in LABELS:
        print(f"\nTop {label} words:\n", top_terms(label, X_tr, vectorizer))
    return X_tr, X_te

# First try: single words, 5,000 most frequent
X_train_tfidf, X_test_tfidf = fit_tfidf(TfidfVectorizer(stop_words="english", max_features=5000))

Train: (19504, 5000) Test: (8359, 5000)

Top negative words:
 tablet    0.041128
amazon    0.038566
slow      0.032128
work      0.029421
buy       0.028687
dtype: float64

Top neutral words:
 tablet    0.055205
good      0.048688
price     0.031577
like      0.030063
ok        0.028105
dtype: float64

Top positive words:
 great     0.054070
love      0.050231
tablet    0.049566
use       0.036806
easy      0.035398
dtype: float64


In [23]:
# Better settings: word pairs, ignore words seen once, softer counts (these are used by the models below)
X_train_tfidf, X_test_tfidf = fit_tfidf(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000, sublinear_tf=True)
)

Train: (19504, 20000) Test: (8359, 20000)

Top negative words:
 amazon      0.025729
tablet      0.024910
slow        0.023243
returned    0.019582
work        0.019275
dtype: float64

Top neutral words:
 tablet    0.032533
good      0.030385
price     0.020123
like      0.018874
ok        0.018398
dtype: float64

Top positive words:
 great     0.030648
love      0.028208
tablet    0.027532
use       0.021104
easy      0.020273
dtype: float64


In [24]:
# Model 1: Naive Bayes (sample weights so rare classes count more)
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train, sample_weight=compute_sample_weight("balanced", y_train))
evaluate("TF-IDF v1 + Naive Bayes", y_test, nb_model.predict(X_test_tfidf))

=== TF-IDF v1 + Naive Bayes
              precision    recall  f1-score   support

    negative       0.18      0.58      0.27       199
     neutral       0.14      0.49      0.21       387
    positive       0.98      0.80      0.88      7773

    accuracy                           0.78      8359
   macro avg       0.43      0.62      0.46      8359
weighted avg       0.92      0.78      0.84      8359

[[ 116   62   21]
 [  92  188  107]
 [ 437 1123 6213]]


In [25]:
# Model 2: Logistic Regression
logreg_v1 = LogisticRegression(class_weight="balanced", max_iter=1000)
logreg_v1.fit(X_train_tfidf, y_train)
evaluate("TF-IDF v1 + LogReg", y_test, logreg_v1.predict(X_test_tfidf))

=== TF-IDF v1 + LogReg
              precision    recall  f1-score   support

    negative       0.29      0.45      0.35       199
     neutral       0.20      0.41      0.27       387
    positive       0.97      0.90      0.93      7773

    accuracy                           0.87      8359
   macro avg       0.49      0.59      0.52      8359
weighted avg       0.92      0.87      0.89      8359

[[  90   49   60]
 [  54  160  173]
 [ 165  584 7024]]


In [26]:
# Model 3: Linear SVM
svm_model = LinearSVC(class_weight="balanced")
svm_model.fit(X_train_tfidf, y_train)
evaluate("TF-IDF v1 + LinearSVC", y_test, svm_model.predict(X_test_tfidf))

=== TF-IDF v1 + LinearSVC
              precision    recall  f1-score   support

    negative       0.38      0.24      0.29       199
     neutral       0.33      0.23      0.27       387
    positive       0.95      0.97      0.96      7773

    accuracy                           0.92      8359
   macro avg       0.55      0.48      0.51      8359
weighted avg       0.91      0.92      0.92      8359

[[  47   32  120]
 [  34   90  263]
 [  43  154 7576]]


## Improving the model

Testing the first saved model in the app showed it handles positive reviews well but fails on negative and neutral ones
(e.g. *"Worst customer service I have ever experienced"* → positive). Reasons:

1. **Very few negative / neutral examples**: 93% of reviews are positive, so words like "price", "quality" and "customer service" mostly appear in positive reviews.
2. **Cleaning removed key words**: the NLTK stopword list drops "not", "no", "too", "very", so *"price is too high"* became *"price high"*.
3. **TF-IDF only counts words**, it does not understand meaning.

What we try, all on the **same test split** (`test_size=0.3`, `random_state=42`, stratified):

| Model | Change |
|---|---|
| TF-IDF v1 (cells above) | stopwords + lemmatizing, text only → macro-F1 ≈ 0.52 (Logistic Regression) |
| **TF-IDF v2** | keep negation words, expand "n't" → "not", no lemmatizing, **title + text**, `C=10` |
| **Pretrained transformer (zero-shot)** | `cardiffnlp/twitter-roberta-base-sentiment-latest`, already trained for negative/neutral/positive, no training on our data |

We compare by **macro-F1** (accuracy hides the small classes) and by a few hand-written reviews.

*Roman Urdu: Har model ko same test set pe macro-F1 se compare karo, phir jeetne wala app mein jayega.*

In [27]:
# Hand-written reviews, including the ones the first app got wrong
HAND_TESTS = [
    "price is too high and quality is really bad",
    "This product broke after two days of light use.",
    "Worst customer service I have ever experienced.",
    "Terrible interface, nothing works as advertised",
    "not going to recommend to anyone",
    "It arrived damaged and missing key parts",
    "It is a standard blue shirt made of cotton",
    "It works, but the screen is dimmer than I expected for the price.",
    "Exceeded my expectations, absolutely love it!",
]

### TF-IDF v2

In [28]:
tfidf_v2 = Pipeline([
    ("clean", FunctionTransformer(text_cleaning.preprocess)),   # keeps not / no / too
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)),
    ("model", LogisticRegression(class_weight="balanced", C=10, max_iter=2000)),
])
tfidf_v2.fit(X_full_train, y_train)          # raw title + text in; the Pipeline cleans it itself
evaluate("TF-IDF v2 + LogReg", y_test, tfidf_v2.predict(X_full_test))

=== TF-IDF v2 + LogReg
              precision    recall  f1-score   support

    negative       0.48      0.51      0.50       199
     neutral       0.35      0.36      0.36       387
    positive       0.97      0.96      0.97      7773

    accuracy                           0.93      8359
   macro avg       0.60      0.61      0.61      8359
weighted avg       0.93      0.93      0.93      8359

[[ 102   44   53]
 [  50  141  196]
 [  61  222 7490]]


### Pretrained transformer (zero-shot)
No training: the model was already trained on ~124M tweets for negative / neutral / positive.
Needs `transformers` and `torch`: if the import fails, run `%pip install transformers torch` and restart the kernel.
The first run downloads the model (~500 MB). Predicting the 8,359 test reviews takes a few minutes on a Mac;
set `EVAL_SAMPLE = 2000` for a quicker (approximate) check.

In [29]:
# Kept here (not at the top) so the notebook still runs if torch is not installed
import torch
from transformers import pipeline as hf_pipeline

TRANSFORMER_ID = "cardiffnlp/twitter-roberta-base-sentiment-latest"
EVAL_SAMPLE = None          # e.g. 2000 for a faster check; None = full test set

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
transformer = hf_pipeline("text-classification", model=TRANSFORMER_ID, device=device)

X_eval, y_eval = X_full_test, y_test
if EVAL_SAMPLE:
    X_eval = X_full_test.sample(EVAL_SAMPLE, random_state=RANDOM_STATE)
    y_eval = y_test.loc[X_eval.index]

out = transformer(X_eval.tolist(), batch_size=32, truncation=True, max_length=512)
evaluate("Transformer zero-shot (roberta)", y_eval, [d["label"].lower() for d in out])

if EVAL_SAMPLE:   # score TF-IDF v2 on the same subset, so the comparison is fair
    evaluate("TF-IDF v2 + LogReg (same sample)", y_eval, tfidf_v2.predict(X_eval))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Transformer zero-shot (roberta)
              precision    recall  f1-score   support

    negative       0.43      0.80      0.56       199
     neutral       0.24      0.11      0.15       387
    positive       0.97      0.97      0.97      7773

    accuracy                           0.93      8359
   macro avg       0.55      0.63      0.56      8359
weighted avg       0.92      0.93      0.92      8359

[[ 160   17   22]
 [ 119   44  224]
 [  94  121 7558]]


### Compare

In [30]:
comparison = pd.DataFrame(results).T.sort_values("macro_f1", ascending=False)
display(comparison)

hand = pd.DataFrame({
    "review": HAND_TESTS,
    "TF-IDF v2": tfidf_v2.predict(HAND_TESTS),
    "Transformer": [d["label"].lower() for d in transformer(HAND_TESTS, truncation=True, max_length=512)],
})
display(hand)

,accuracy,macro_f1,f1_negative,f1_neutral,f1_positive
TF-IDF v2 + LogReg,0.925,0.605,0.495,0.355,0.966
Transformer zero-shot (roberta),0.929,0.562,0.559,0.155,0.970
TF-IDF v1 + LogReg,0.870,0.520,0.354,0.271,0.935
TF-IDF v1 + LinearSVC,0.923,0.509,0.291,0.271,0.963
TF-IDF v1 + Naive Bayes,0.780,0.456,0.275,0.214,0.880


,review,TF-IDF v2,Transformer
0,price is too high and quality is really bad,positive,negative
1,This product broke after two days of light use.,negative,negative
2,Worst customer service I have ever experienced.,positive,negative
3,"Terrible interface, nothing works as advertised",neutral,negative
4,not going to recommend to anyone,positive,negative
5,It arrived damaged and missing key parts,negative,negative
6,It is a standard blue shirt made of cotton,neutral,neutral
7,"It works, but the screen is dimmer than I expe...",neutral,neutral
8,"Exceeded my expectations, absolutely love it!",positive,positive


### Save the results and our model (locally)
Both models are offered in the Streamlit app (model picker), so we do not choose one here.
This cell **only saves files on your computer**, nothing is uploaded:

1. `outputs/metrics/sentiment_model_comparison.csv`: the comparison table
2. `models/tfidf-v2/`: our model folder

| File | Why |
|---|---|
| `sentiment_model.joblib` | our trained TF-IDF v2 + LogReg Pipeline (the Streamlit app downloads exactly this name) |
| `text_cleaning.py` | the Pipeline imports it, so it must travel with the model |
| `requirements.txt` | exact versions (scikit-learn must match when loading) |
| `README.md` | model card with our real test scores |

RoBERTa is not saved: the app downloads it directly from `cardiffnlp` on Hugging Face.

To publish, upload the folder yourself to https://huggingface.co/hamza-shabbir94/amazon-review-sentiment-analysis
(e.g. from the project root: `hf upload hamza-shabbir94/amazon-review-sentiment-analysis models/tfidf-v2 .`).

*Roman Urdu: Yeh cell sirf aap ke computer pe `models/tfidf-v2/` mein model save karti hai. Hugging Face pe upload aap khud karenge.*

In [ ]:
# Save the comparison table and our model LOCALLY in models/tfidf-v2/
# (nothing is uploaded here; upload this folder to Hugging Face yourself)
import shutil

# 1) comparison table
comparison.to_csv(config.METRICS_DIR / "sentiment_model_comparison.csv")

# 2) our model + the cleaning code it needs
MODEL_DIR = config.MODELS_DIR / "tfidf-v2"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(tfidf_v2, MODEL_DIR / "sentiment_model.joblib")
shutil.copy(text_cleaning.__file__, MODEL_DIR / "text_cleaning.py")

# 3) exact library versions
(MODEL_DIR / "requirements.txt").write_text(
    f"scikit-learn=={sklearn.__version__}\n"
    f"nltk=={nltk.__version__}\n"
    f"joblib=={joblib.__version__}\n"
)

# 4) model card (README.md) filled with our real scores on the full test set
m = results["TF-IDF v2 + LogReg"]
readme = f"""---
library_name: sklearn
pipeline_tag: text-classification
language: en
tags:
  - sentiment-analysis
  - tfidf
  - logistic-regression
  - amazon-reviews
---

# Amazon Review Sentiment: TF-IDF + Logistic Regression

A small, fast sentiment classifier for product reviews: **negative / neutral / positive**.
Built by Hamza Shabbir & Yasaman Najafi Jozani (Ironhack AI Engineering, Project 4).

## What is inside

| File | What it is |
|---|---|
| `sentiment_model.joblib` | scikit-learn `Pipeline`: text cleaning -> TF-IDF (1-2 word n-grams, 50,000 features) -> Logistic Regression (`class_weight="balanced"`, `C=10`) |
| `text_cleaning.py` | the cleaning step (lowercase, letters only, stopwords removed **but "not", "no", "too" kept**) |
| `requirements.txt` | library versions used for training |

## Training data

Consumer Reviews of Amazon Products (Datafiniti, `1429_1.csv`), {len(pre_processed):,} cleaned reviews.
Input = review title + ". " + review text. Labels: 1-2 stars negative, 3 neutral, 4-5 positive.
Split {1 - TEST_SIZE:.0%} train / {TEST_SIZE:.0%} test, stratified, `random_state={RANDOM_STATE}`.

## Results ({len(X_full_test):,} test reviews)

| Metric | Score |
|---|---|
| Accuracy | {m["accuracy"]} |
| **Macro-F1** | **{m["macro_f1"]}** |
| F1 negative | {m["f1_negative"]} |
| F1 neutral | {m["f1_neutral"]} |
| F1 positive | {m["f1_positive"]} |

## How to use

```python
import os, sys, joblib
from huggingface_hub import hf_hub_download

REPO = "hamza-shabbir94/amazon-review-sentiment-analysis"
model_path = hf_hub_download(REPO, "sentiment_model.joblib")
clean_path = hf_hub_download(REPO, "text_cleaning.py")
sys.path.insert(0, os.path.dirname(clean_path))   # the pipeline imports text_cleaning
model = joblib.load(model_path)
print(model.predict(["Battery died after two days, very disappointed."]))
```

## Limitations

- Trained almost only on Amazon devices (Kindle, Fire, Echo).
- Weak on neutral (mixed 3-star) reviews. A pretrained transformer
  (`cardiffnlp/twitter-roberta-base-sentiment-latest`) handles unseen negative wording better.
- `.joblib` files are Python pickles: only load them from sources you trust.
"""
(MODEL_DIR / "README.md").write_text(readme)

print("Saved locally:")
print(" ", config.METRICS_DIR / "sentiment_model_comparison.csv")
for f in sorted(MODEL_DIR.iterdir()):
    print(f"  {f}  ({f.stat().st_size / 1024:.1f} KB)")
